# SmartRoad AI — YOLO11 Pothole Detector Training
### Google Colab Free T4 GPU Pipeline

This notebook trains a custom **YOLO11n** model specifically on the **RDD2022** dataset (targeting class **D40: Pothole**) to produce `models/pothole_best.pt` for the SmartRoad AI platform.

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# 2. Install Ultralytics and dependencies
!pip install -q ultralytics roboflow

### 3. Download RDD2022 Dataset Subset
We use the official RDD2022 benchmark subset (India / Japan) which has dense pothole annotations (D40).

In [ ]:
import os
!mkdir -p rdd2022_raw
%cd rdd2022_raw

# Download India subset (~500MB) from Figshare / Sekilab mirror
!wget -O RDD2022_India.zip https://figshare.com/ndownloader/files/37943568
!unzip -q RDD2022_India.zip
%cd ..

### 4. Convert Pascal VOC XML Annotations to YOLO Format (Class D40)

In [ ]:
import glob, xml.etree.ElementTree as ET, shutil, random

output_dir = "datasets/rdd2022_potholes"
for s in ['train', 'val', 'test']:
    os.makedirs(f"{output_dir}/images/{s}", exist_ok=True)
    os.makedirs(f"{output_dir}/labels/{s}", exist_ok=True)

xmls = glob.glob("rdd2022_raw/**/xmls/*.xml", recursive=True)
print(f"Found {len(xmls)} raw annotations.")

pothole_samples = []
for xml_path in xmls:
    tree = ET.parse(xml_path)
    root = tree.getroot()
    size = root.find("size")
    if size is None: continue
    w, h = int(size.find("width").text), int(size.find("height").text)
    if w <= 0 or h <= 0: continue
    
    boxes = []
    for obj in root.findall("object"):
        if obj.find("name").text == "D40":  # Pothole
            box = obj.find("bndbox")
            xmin = float(box.find("xmin").text)
            ymin = float(box.find("ymin").text)
            xmax = float(box.find("xmax").text)
            ymax = float(box.find("ymax").text)
            xc = (xmin + xmax) / (2.0 * w)
            yc = (ymin + ymax) / (2.0 * h)
            bw = (xmax - xmin) / float(w)
            bh = (ymax - ymin) / float(h)
            boxes.append((0, xc, yc, bw, bh))
            
    if boxes:
        base = os.path.splitext(os.path.basename(xml_path))[0]
        img_path = xml_path.replace("/xmls/", "/images/").replace(".xml", ".jpg")
        if os.path.exists(img_path):
            pothole_samples.append((img_path, boxes))

print(f"Extracted {len(pothole_samples)} confirmed pothole images.")
random.seed(42)
random.shuffle(pothole_samples)
n = len(pothole_samples)
splits = {
    'train': pothole_samples[:int(n*0.8)],
    'val': pothole_samples[int(n*0.8):int(n*0.9)],
    'test': pothole_samples[int(n*0.9):]
}

for s, items in splits.items():
    for img_p, boxes in items:
        b = os.path.basename(img_p)
        shutil.copy2(img_p, f"{output_dir}/images/{s}/{b}")
        with open(f"{output_dir}/labels/{s}/{b.replace('.jpg', '.txt')}", 'w') as lf:
            for c, xc, yc, bw, bh in boxes:
                lf.write(f"{c} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

yaml_str = f"""path: {os.path.abspath(output_dir)}
train: images/train
val: images/val
test: images/test
names:
  0: pothole
"""
with open(f"{output_dir}/data.yaml", 'w') as yf:
    yf.write(yaml_str)
print("YOLO Dataset ready at datasets/rdd2022_potholes/data.yaml")

### 5. Train YOLO11n on GPU

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
results = model.train(
    data="datasets/rdd2022_potholes/data.yaml",
    epochs=40,
    imgsz=640,
    batch=16,
    device=0,
    patience=10,
    save=True,
    project="runs",
    name="smartroad_pothole"
)

### 6. Evaluate Model on Test Set

In [ ]:
val_metrics = model.val(split="test")
print("=== SmartRoad AI Model Evaluation Results ===")
print(f"mAP@50:    {val_metrics.box.map50:.4f}")
print(f"mAP@50-95: {val_metrics.box.map:.4f}")
print(f"Precision: {val_metrics.box.mp:.4f}")
print(f"Recall:    {val_metrics.box.mr:.4f}")

### 7. Download `pothole_best.pt`
Place this downloaded file into your local `models/pothole_best.pt` in the SmartRoad-AI workspace.

In [ ]:
from google.colab import files
files.download("runs/smartroad_pothole/weights/best.pt")